# 03 — Machine Learning Models for FIQA Score Prediction

This notebook trains and compares regression models that predict **CR-FIQA scores** from facial attributes and demographic information.

It continues the workflow from the previous notebooks:

1. `00_colab_setup.ipynb` prepares the shared project environment.
2. `01_extract_and_merge_cr_fiqa_scores.ipynb` creates `diveface_fiqa_merged.csv`.
3. `02_exploratory_data_analysis.ipynb` examines distributions, correlations, and demographic groups.
4. **This notebook** performs identity-aware model training, evaluation, interpretation, and artifact export.

The notebook evaluates:

- Mean baseline
- Linear Regression
- Ridge Regression
- Lasso Regression
- Random Forest Regression
- Gradient Boosting Regression
- XGBoost Regression

To avoid identity leakage, all images belonging to the same identity (`cls`) are assigned exclusively to either the training or test set.


## 1. Shared project setup


In [ ]:
# Locate and run the shared setup notebook.
#
# Recommended repository layout:
# fiqa-demographic-analysis/
# └── notebooks/
#     ├── 00_colab_setup.ipynb
#     ├── 01_extract_and_merge_cr_fiqa_scores.ipynb
#     ├── 02_exploratory_data_analysis.ipynb
#     └── 03_ml_models.ipynb

from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(f"- {path}" for path in setup_candidates)
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep the notebooks in the same notebooks/ folder or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")
get_ipython().run_line_magic("run", f'"{SETUP_NOTEBOOK}"')


## 2. Imports and output paths


In [ ]:
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "XGBoost is required for this notebook. Install it with "
        "`pip install xgboost` and rerun the cell."
    ) from exc

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "cr_fiqa_score"
IDENTITY_COLUMN = "cls"

MERGED_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"
RESULTS_PATH = PROJECT_PATH / "results" / "03_ml_models"
FIGURES_PATH = RESULTS_PATH / "figures"
TABLES_PATH = RESULTS_PATH / "tables"
MODELS_PATH = PROJECT_PATH / "models" / "03_ml_models"

for path in [RESULTS_PATH, FIGURES_PATH, TABLES_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

if not MERGED_FILE.exists():
    raise FileNotFoundError(
        f"Merged dataset was not found: {MERGED_FILE}\n"
        "Run 01_extract_and_merge_cr_fiqa_scores.ipynb first."
    )

print(f"Merged dataset: {MERGED_FILE}")
print(f"Results folder: {RESULTS_PATH}")
print(f"Models folder:  {MODELS_PATH}")


## 3. Load and validate the modeling dataset

Only a compact validation is repeated here. Detailed descriptive analysis belongs to Notebook 02.


In [ ]:
df = pd.read_csv(MERGED_FILE)

print(f"Dataset shape: {df.shape}")
print(f"Number of identities: {df[IDENTITY_COLUMN].nunique()}")
print(f"Missing target values: {df[TARGET].isna().sum()}")

display(df.head(3))


## 4. Define model features

The feature grouping follows the annotation semantics used throughout the project:

- **Continuous features:** numerical measurements or graded attributes.
- **Binary features:** indicators describing whether an attribute is present.
- **Categorical features:** demographic group labels encoded using one-hot encoding.

`index` is retained only as dataset metadata, while `cls` is used exclusively for the identity-aware split. Neither is used as a predictor.


In [ ]:
continuous_features = [
    "age",
    "smile",
    "moustache",
    "beard",
    "sideburns",
    "head_roll",
    "head_yaw",
    "head_pitch",
    "blur",
    "exposure",
    "noise",
]

binary_features = [
    "mask",
    "headWear",
    "glasses",
    "eye_makeup",
    "lip_makeup",
    "forehead_occluded",
    "eye_occluded",
    "mouth_occluded",
]

categorical_features = [
    "group",
]

feature_columns = (
    continuous_features
    + binary_features
    + categorical_features
)

required_columns = set(
    feature_columns
    + [TARGET, IDENTITY_COLUMN, "index"]
)
missing_columns = sorted(required_columns.difference(df.columns))

if missing_columns:
    raise KeyError(
        "The merged dataset is missing required columns: "
        f"{missing_columns}"
    )

print(f"Continuous features:  {len(continuous_features)}")
print(f"Binary features:      {len(binary_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total model features: {len(feature_columns)}")


## 5. Prepare model-ready rows


In [ ]:
model_df = df[
    ["index", IDENTITY_COLUMN, TARGET] + feature_columns
].copy()

# Convert binary annotations to numeric values without changing their meaning.
for feature in binary_features:
    model_df[feature] = pd.to_numeric(
        model_df[feature],
        errors="coerce",
    )

# Continuous values must be numeric for scaling and regression.
for feature in continuous_features + [TARGET]:
    model_df[feature] = pd.to_numeric(
        model_df[feature],
        errors="coerce",
    )

missing_by_column = model_df.isna().sum()
missing_by_column = missing_by_column[missing_by_column > 0]

if not missing_by_column.empty:
    print("Rows with missing model values will be removed:")
    display(missing_by_column.to_frame("missing_values"))

rows_before = len(model_df)
model_df = model_df.dropna(
    subset=[IDENTITY_COLUMN, TARGET] + feature_columns
).copy()
rows_removed = rows_before - len(model_df)

if model_df.empty:
    raise RuntimeError("No complete rows remain for model training.")

print(f"Rows retained: {len(model_df):,}")
print(f"Rows removed:  {rows_removed:,}")


## 6. Identity-aware train-test split

A standard random split could place different images of the same identity in both subsets. This would create identity leakage and could lead to overly optimistic test performance.

`GroupShuffleSplit` therefore uses `cls` as the grouping variable. The split is fixed with `random_state=42` and saved for all later notebooks.


In [ ]:
X = model_df[feature_columns].copy()
y = model_df[TARGET].copy()
groups = model_df[IDENTITY_COLUMN].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_positions, test_positions = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_positions].copy()
X_test = X.iloc[test_positions].copy()
y_train = y.iloc[train_positions].copy()
y_test = y.iloc[test_positions].copy()

groups_train = groups.iloc[train_positions]
groups_test = groups.iloc[test_positions]

identity_overlap = set(groups_train).intersection(set(groups_test))
if identity_overlap:
    raise RuntimeError(
        f"Identity leakage detected for {len(identity_overlap)} identities."
    )

print(f"Training rows:       {len(X_train):,}")
print(f"Test rows:           {len(X_test):,}")
print(f"Training identities: {groups_train.nunique():,}")
print(f"Test identities:     {groups_test.nunique():,}")
print("Identity overlap:    0")


## 7. Preprocessing pipeline

The same preprocessing structure is fitted independently inside every model pipeline:

- continuous features are standardized;
- binary indicators pass through unchanged;
- the demographic group is one-hot encoded;
- unknown categories at test time are ignored safely.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor


## 8. Define regression models

The initial parameter settings provide a transparent and reproducible comparison. Hyperparameter optimization is intentionally kept separate from this baseline model notebook.


In [ ]:
regressors = {
    "Mean Baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(
        alpha=0.001,
        max_iter=10_000,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        min_samples_split=5,
        min_samples_leaf=2,
        loss="squared_error",
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

model_pipelines = {
    model_name: Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("regressor", regressor),
        ]
    )
    for model_name, regressor in regressors.items()
}

print("Models to evaluate:")
for model_name in model_pipelines:
    print(f"- {model_name}")


## 9. Train and evaluate all models


In [ ]:
def regression_metrics(y_true, y_pred):
    """Return MAE, RMSE, and R² for a regression prediction."""
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
    }


evaluation_rows = []
predictions = {}
fitted_models = {}

for model_name, pipeline in model_pipelines.items():
    print(f"Training {model_name}...")
    start_time = time.perf_counter()

    pipeline.fit(X_train, y_train)

    train_predictions = pipeline.predict(X_train)
    test_predictions = pipeline.predict(X_test)
    elapsed_seconds = time.perf_counter() - start_time

    train_metrics = regression_metrics(y_train, train_predictions)
    test_metrics = regression_metrics(y_test, test_predictions)

    evaluation_rows.append(
        {
            "model": model_name,
            "train_mae": train_metrics["mae"],
            "test_mae": test_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "test_rmse": test_metrics["rmse"],
            "train_r2": train_metrics["r2"],
            "test_r2": test_metrics["r2"],
            "training_seconds": elapsed_seconds,
        }
    )

    predictions[model_name] = test_predictions
    fitted_models[model_name] = pipeline

model_comparison = (
    pd.DataFrame(evaluation_rows)
    .sort_values("test_rmse")
    .reset_index(drop=True)
)

display(model_comparison.round(4))


## 10. Select the best model


In [ ]:
best_model_name = model_comparison.loc[0, "model"]
best_model = fitted_models[best_model_name]
best_test_rmse = model_comparison.loc[0, "test_rmse"]
best_test_mae = model_comparison.loc[0, "test_mae"]
best_test_r2 = model_comparison.loc[0, "test_r2"]

print(f"Best model: {best_model_name}")
print(f"Test RMSE:  {best_test_rmse:.4f}")
print(f"Test MAE:   {best_test_mae:.4f}")
print(f"Test R²:    {best_test_r2:.4f}")


## 11. Model comparison plots


In [ ]:
def save_metric_plot(
    comparison_table,
    metric,
    title,
    ylabel,
    ascending,
    filename,
):
    plot_data = comparison_table.sort_values(
        metric,
        ascending=ascending,
    )

    plt.figure(figsize=(10, 5))
    plt.bar(plot_data["model"], plot_data[metric])
    plt.xlabel("Model")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / filename, dpi=300, bbox_inches="tight")
    plt.show()


save_metric_plot(
    model_comparison,
    metric="test_rmse",
    title="Test RMSE by Regression Model",
    ylabel="RMSE",
    ascending=True,
    filename="model_comparison_test_rmse.png",
)

save_metric_plot(
    model_comparison,
    metric="test_mae",
    title="Test MAE by Regression Model",
    ylabel="MAE",
    ascending=True,
    filename="model_comparison_test_mae.png",
)

save_metric_plot(
    model_comparison,
    metric="test_r2",
    title="Test R² by Regression Model",
    ylabel="R²",
    ascending=False,
    filename="model_comparison_test_r2.png",
)


## 12. Best-model prediction diagnostics


In [ ]:
best_predictions = predictions[best_model_name]

minimum_value = min(y_test.min(), best_predictions.min())
maximum_value = max(y_test.max(), best_predictions.max())

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_predictions, alpha=0.4)
plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
)
plt.xlabel("Actual CR-FIQA Score")
plt.ylabel("Predicted CR-FIQA Score")
plt.title(f"Actual vs. Predicted Scores — {best_model_name}")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "best_model_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

residuals = y_test.to_numpy() - best_predictions

plt.figure(figsize=(8, 5))
plt.scatter(best_predictions, residuals, alpha=0.4)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted CR-FIQA Score")
plt.ylabel("Residual: Actual − Predicted")
plt.title(f"Residual Plot — {best_model_name}")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "best_model_residuals.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 13. Model interpretation

Linear-model coefficients and tree-based feature importances are exported separately because they represent different quantities. They should not be interpreted as equivalent effect estimates.


In [ ]:
def get_processed_feature_names(fitted_pipeline):
    return fitted_pipeline.named_steps[
        "preprocessor"
    ].get_feature_names_out()


def coefficient_table(fitted_pipeline):
    feature_names = get_processed_feature_names(fitted_pipeline)
    coefficients = fitted_pipeline.named_steps["regressor"].coef_

    table = pd.DataFrame(
        {
            "feature": feature_names,
            "coefficient": coefficients,
        }
    )
    table["absolute_coefficient"] = table["coefficient"].abs()
    return table.sort_values(
        "absolute_coefficient",
        ascending=False,
    ).reset_index(drop=True)


def importance_table(fitted_pipeline):
    feature_names = get_processed_feature_names(fitted_pipeline)
    importances = fitted_pipeline.named_steps[
        "regressor"
    ].feature_importances_

    return (
        pd.DataFrame(
            {
                "feature": feature_names,
                "importance": importances,
            }
        )
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


linear_tables = {
    model_name: coefficient_table(fitted_models[model_name])
    for model_name in [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
    ]
}

tree_tables = {
    model_name: importance_table(fitted_models[model_name])
    for model_name in [
        "Random Forest",
        "Gradient Boosting",
        "XGBoost",
    ]
}

print("Top Random Forest feature importances:")
display(tree_tables["Random Forest"].head(15).round(4))

print("Largest absolute Linear Regression coefficients:")
display(linear_tables["Linear Regression"].head(15).round(4))


In [ ]:
top_importances = tree_tables["Random Forest"].head(15).sort_values(
    "importance",
    ascending=True,
)

plt.figure(figsize=(9, 6))
plt.barh(top_importances["feature"], top_importances["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest — Top 15 Feature Importances")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "random_forest_top_feature_importances.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 14. Save tables, predictions, split information, and models


In [ ]:
# Model comparison table
model_comparison.to_csv(
    TABLES_PATH / "model_comparison.csv",
    index=False,
)

# Test predictions plus metadata needed for later demographic analyses
prediction_table = model_df.iloc[test_positions][
    ["index", IDENTITY_COLUMN, "group", TARGET]
].copy()
prediction_table = prediction_table.rename(
    columns={TARGET: "actual_cr_fiqa_score"}
)

for model_name, model_predictions in predictions.items():
    prediction_column = (
        model_name.lower()
        .replace(" ", "_")
        .replace("²", "2")
        + "_prediction"
    )
    prediction_table[prediction_column] = model_predictions

prediction_table.to_csv(
    TABLES_PATH / "model_predictions.csv",
    index=False,
)

# Exact dataset row indices and identity-aware split labels
split_information = model_df[["index", IDENTITY_COLUMN]].copy()
split_information["source_dataframe_index"] = model_df.index
split_information["split"] = "train"
split_information.iloc[test_positions, split_information.columns.get_loc("split")] = "test"
split_information.to_csv(
    TABLES_PATH / "identity_aware_split.csv",
    index=False,
)

np.save(
    RESULTS_PATH / "train_indices.npy",
    model_df.index.to_numpy()[train_positions],
)
np.save(
    RESULTS_PATH / "test_indices.npy",
    model_df.index.to_numpy()[test_positions],
)

# Coefficients and feature importances
for model_name, table in linear_tables.items():
    filename = model_name.lower().replace(" ", "_") + "_coefficients.csv"
    table.to_csv(TABLES_PATH / filename, index=False)

for model_name, table in tree_tables.items():
    filename = model_name.lower().replace(" ", "_") + "_feature_importances.csv"
    table.to_csv(TABLES_PATH / filename, index=False)

# Save complete fitted pipelines, including preprocessing.
for model_name, fitted_pipeline in fitted_models.items():
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(fitted_pipeline, MODELS_PATH / filename)

joblib.dump(best_model, MODELS_PATH / "best_model.joblib")

run_metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target": TARGET,
    "identity_column": IDENTITY_COLUMN,
    "feature_columns": feature_columns,
    "best_model": best_model_name,
    "best_test_rmse": float(best_test_rmse),
    "best_test_mae": float(best_test_mae),
    "best_test_r2": float(best_test_r2),
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "training_identities": int(groups_train.nunique()),
    "test_identities": int(groups_test.nunique()),
}

with (RESULTS_PATH / "run_metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(run_metadata, file, indent=2)

print("All model artifacts were saved successfully.")


## 15. Saved artifact summary


In [ ]:
print("Saved result files:")
for file_path in sorted(RESULTS_PATH.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(RESULTS_PATH))

print("\nSaved model files:")
for file_path in sorted(MODELS_PATH.glob("*.joblib")):
    print("-", file_path.name)


## 16. Conclusion

This notebook establishes a reproducible baseline for predicting CR-FIQA scores from facial attributes and demographic group information.

The main outputs are:

- an identity-disjoint training and test split;
- a comparison of linear, regularized linear, ensemble, and boosting models;
- test-set predictions for later error and demographic analyses;
- coefficients and feature-importance tables for model interpretation;
- complete fitted pipelines and the selected best model;
- saved train/test indices for consistent reuse in subsequent notebooks.

The next notebooks can build on these artifacts for hyperparameter optimization, correlation and regression analyses, final model evaluation, and demographic consistency analysis without recreating the split.
